In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D,
    Dense, Flatten
)
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [ ]:
STYLE_MAP = {
    "Back": 0,
    "Front": 1,
    "Goblet": 2
}
NUM_CLASSES = 3

DATA_ROOT = "..."

X, y = [], []

for root, _, files in os.walk(DATA_ROOT):
    for file in files:
        if not file.endswith(".npy"):
            continue

        full_path = os.path.join(root, file)

        style_label = None
        for style, label_id in STYLE_MAP.items():
            if f"/{style}/" in full_path:
                style_label = label_id
                break

        if style_label is None:
            continue

        data = np.load(full_path)
        data = data.reshape(50, 34)

        X.append(data)
        y.append(style_label)

X = np.array(X)
y = to_categorical(y, NUM_CLASSES)

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y.argmax(axis=1)
)


In [ ]:
input_layer = Input(shape=(50, 34))

x = Conv1D(32, kernel_size=3, activation="relu")(input_layer)
x = MaxPooling1D(pool_size=2)(x)
x = Flatten()(x)
x = Dense(32, activation="relu")(x)

output = Dense(NUM_CLASSES, activation="softmax")(x)

model_cnn = Model(input_layer, output)

In [ ]:
model_cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_cnn.summary()

In [ ]:
history_cnn = model_cnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=16,
    verbose=1
)


In [ ]:
model_cnn.save(
    "1D_CNN_Style.keras"
)
